# NCHS - Drug Poisoning Mortality by State

This dataset describes drug poisoning deaths at the U.S. and state level by selected demographic characteristics, and includes age-adjusted death rates for drug poisoning.

In [1]:
import pandas as pd
import numpy as np
from sodapy import Socrata
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# API Configuration
DATASET_ID = "xbxb-epbu"
DOMAIN = "data.cdc.gov"
APP_TOKEN = "CpwHhA1scJ9x7VNZ3PWM8MRG6"

# Initialize client
client = Socrata(DOMAIN, APP_TOKEN)

# Fetch the data
results = client.get(DATASET_ID, limit=5000)

# Convert to pandas DataFrame
df = pd.DataFrame.from_records(results)

df.head()

,state,year,sex,age_group,race_and_hispanic_origin,deaths,population,crude_death_rate,standard_error_for_crude_rate,lower_confidence_limit_for_crude_rate,upper_confidence_limit_for_crude_rate,age_adjusted_rate,standard_error_for_age_adjusted_rate,lower_confidence_limit_for_age_adjusted_rate,upper_confidence_limit_for_age_adjusted_rate,state_crude_rate_in_range,us_crude_rate,us_age_adjusted_rate,unit
0,Alabama,1999,Both Sexes,All Ages,All Races-All Origins,169,4430143,3.8148,0.29344,3.2396,4.3899,3.8521,0.29657,3.2708,4.4334,1.8–7.1,6.0382,6.0570,"per 100,000 population"
1,Alabama,2000,Both Sexes,All Ages,All Races-All Origins,197,4447100,4.4299,0.31561,3.8112,5.0485,4.4857,0.31985,3.8588,5.1126,1.8–7.1,6.1882,6.1749,"per 100,000 population"
2,Alabama,2001,Both Sexes,All Ages,All Races-All Origins,216,4467634,4.8348,0.32896,4.1900,5.4795,4.8915,0.33329,4.2382,5.5447,1.8–7.1,6.8057,6.7922,"per 100,000 population"
3,Alabama,2002,Both Sexes,All Ages,All Races-All Origins,211,4480089,4.7097,0.32423,4.0742,5.3452,4.7619,0.32868,4.1177,5.4062,1.8–7.1,8.1766,8.1957,"per 100,000 population"
4,Alabama,2003,Both Sexes,All Ages,All Races-All Origins,197,4503491,4.3744,0.31166,3.7635,4.9852,4.4333,0.31701,3.8120,5.0547,1.8–7.1,8.8881,8.8765,"per 100,000 population"


**Missing Values**

The data is missing values for `age_adjusted_rate`, `standard_error_for_age_adjusted_rate`, `lower_confidence_limit_for_age_adjusted_rate`,
`upper_confidence_limit_for_age_adjusted_rate`, and `state_crude_rate_in_range` for rows where `age_group` is not "All Ages". When analyzing age-specific data, we will use the crude death rate instead. For data completeness, we will substitute the crude rate for missing rates with the understanding that the columns for age-adjusted rate are only to be used when looking at All Ages. We will drop the range column because its values are inconsistent and we can recreate as needed.

We will also create a new set of columns for death rate that combines the crude and age-adjusted rates based on age group.

In [3]:
# Drop range column
df.drop('state_crude_rate_in_range', axis=1, inplace=True)
# Fill Missing Values
df['age_adjusted_rate'] = df['age_adjusted_rate'].fillna(df['crude_death_rate'])
df['standard_error_for_age_adjusted_rate'] = df['standard_error_for_age_adjusted_rate'].fillna(df['standard_error_for_crude_rate'])
df['lower_confidence_limit_for_age_adjusted_rate'] = df['lower_confidence_limit_for_age_adjusted_rate'].fillna(df['lower_confidence_limit_for_crude_rate'])
df['upper_confidence_limit_for_age_adjusted_rate'] = df['upper_confidence_limit_for_age_adjusted_rate'].fillna(df['upper_confidence_limit_for_crude_rate'])

In [4]:
# Combine crude and age-adjusted rates to one column for ease of future analysis
df['death_rate'] = np.where(df.age_group=='All Ages', df.age_adjusted_rate, df.crude_death_rate)
df['standard_error_death_rate'] = np.where(df.age_group=='All Ages', df.standard_error_for_age_adjusted_rate, df.standard_error_for_crude_rate)
df['lower_confidence_death_rate'] = np.where(df.age_group=='All Ages', df.lower_confidence_limit_for_age_adjusted_rate, df.lower_confidence_limit_for_crude_rate)
df['upper_confidence_death_rate'] = np.where(df.age_group=='All Ages', df.upper_confidence_limit_for_age_adjusted_rate, df.upper_confidence_limit_for_crude_rate)

**Race and Hispanic Origin**

Rename categories for brevity.

In [5]:
# Rename column
df.rename(columns={'race_and_hispanic_origin':'race'}, inplace=True)

# Mapping for Race & Hispanic Origin column
race_map = {'All Races-All Origins': 'All Races',
           'Hispanic': 'Hispanic',
           'Non-Hispanic Black':'Black',
           'Non-Hispanic White': 'White'}

df.race = df.race.map(race_map)

In [6]:
df.columns

Index(['state', 'year', 'sex', 'age_group', 'race', 'deaths', 'population',
       'crude_death_rate', 'standard_error_for_crude_rate',
       'lower_confidence_limit_for_crude_rate',
       'upper_confidence_limit_for_crude_rate', 'age_adjusted_rate',
       'standard_error_for_age_adjusted_rate',
       'lower_confidence_limit_for_age_adjusted_rate',
       'upper_confidence_limit_for_age_adjusted_rate', 'us_crude_rate',
       'us_age_adjusted_rate', 'unit', 'death_rate',
       'standard_error_death_rate', 'lower_confidence_death_rate',
       'upper_confidence_death_rate'],
      dtype='object')

**Keep Relevant Columns**

In [7]:
cols_to_keep = ['state', 'year', 'sex', 'age_group', 'race',
 'death_rate', 'standard_error_death_rate',
 'lower_confidence_death_rate', 'upper_confidence_death_rate',
'us_crude_rate', 'us_age_adjusted_rate','unit']

df = df[cols_to_keep]
df.head()

,state,year,sex,age_group,race,death_rate,standard_error_death_rate,lower_confidence_death_rate,upper_confidence_death_rate,us_crude_rate,us_age_adjusted_rate,unit
0,Alabama,1999,Both Sexes,All Ages,All Races,3.8521,0.29657,3.2708,4.4334,6.0382,6.0570,"per 100,000 population"
1,Alabama,2000,Both Sexes,All Ages,All Races,4.4857,0.31985,3.8588,5.1126,6.1882,6.1749,"per 100,000 population"
2,Alabama,2001,Both Sexes,All Ages,All Races,4.8915,0.33329,4.2382,5.5447,6.8057,6.7922,"per 100,000 population"
3,Alabama,2002,Both Sexes,All Ages,All Races,4.7619,0.32868,4.1177,5.4062,8.1766,8.1957,"per 100,000 population"
4,Alabama,2003,Both Sexes,All Ages,All Races,4.4333,0.31701,3.8120,5.0547,8.8881,8.8765,"per 100,000 population"
